In [0]:
# =============================================================
# CREDIT-POLICY ROUTING TOOL
#
# Requires : 10_runtime_config + 30_credit_risk_model_tool
# Globals used: DOCUMENT_FACTS_FILE, DOCUMENT_MANIFEST_FILE,
#               POLICY_VERSION, calculate_credit_risk()
# Exposes  : apply_credit_policy(borrower_id) -> dict
# =============================================================

from typing import Any
import pandas as pd

# DOCUMENT_FACTS_FILE, DOCUMENT_MANIFEST_FILE and POLICY_VERSION
# are in scope from 10_runtime_config via %run.

REQUIRED_DOCUMENT_TYPES = {
    "loan_application",
    "financial_summary",
    "credit_analyst_note",
}

# ---------------------------------------------------------
# Load policy inputs once per session
# ---------------------------------------------------------
document_facts_df    = pd.read_parquet(DOCUMENT_FACTS_FILE)
document_manifest_df = pd.read_parquet(DOCUMENT_MANIFEST_FILE)

assert document_facts_df["borrower_id"].is_unique
assert document_manifest_df["document_id"].is_unique

# Prevent future-outcome leakage into policy routing.
FORBIDDEN_POLICY_COLUMNS = {
    "bankrupt_within_1_year",
    "demo_risk_group",
    "selection_stress_score",
}
assert not FORBIDDEN_POLICY_COLUMNS.intersection(
    document_facts_df.columns
), "Outcome leakage detected in policy facts."


# ---------------------------------------------------------
# Routing function
# ---------------------------------------------------------
def apply_credit_policy(
    borrower_id: str,
) -> dict[str, Any]:
    """
    Apply transparent prototype workflow rules.

    Chooses the required analyst-review route.
    Does not approve, reject, or price a loan.
    """
    if not isinstance(borrower_id, str):
        raise TypeError("borrower_id must be a string.")

    borrower_id = borrower_id.strip().upper()

    if not borrower_id.startswith("B"):
        raise ValueError("borrower_id must use B-format, for example B000187.")

    # A. Governed model result
    risk_result = calculate_credit_risk(borrower_id)

    # B. Permitted financial facts
    borrower_facts = document_facts_df[
        document_facts_df["borrower_id"] == borrower_id
    ]
    if borrower_facts.empty:
        raise ValueError(f"No document facts found for {borrower_id}.")
    if len(borrower_facts) != 1:
        raise RuntimeError(f"Multiple fact records found for {borrower_id}.")
    facts = borrower_facts.iloc[0]

    # C. Validate required documents
    borrower_documents = document_manifest_df[
        document_manifest_df["borrower_id"] == borrower_id
    ]
    document_counts          = borrower_documents["document_type"].value_counts().to_dict()
    present_document_types   = set(document_counts.keys())
    missing_document_types   = sorted(REQUIRED_DOCUMENT_TYPES - present_document_types)
    duplicate_document_types = sorted([
        dt for dt, count in document_counts.items() if count > 1
    ])

    # D. Apply transparent rules
    triggered_rules: list[dict] = []

    def add_rule(rule_id: str, severity: str, category: str, message: str) -> None:
        triggered_rules.append({
            "rule_id":  rule_id, "severity": severity,
            "category": category, "message":  message,
        })

    if missing_document_types:
        add_rule(
            "DOC-001", "BLOCKING", "DOCUMENT_COMPLETENESS",
            "Required documents are missing: " + ", ".join(missing_document_types),
        )
    if duplicate_document_types:
        add_rule(
            "DOC-002", "BLOCKING", "DOCUMENT_INTEGRITY",
            "Duplicate document types were found: " + ", ".join(duplicate_document_types),
        )
    if risk_result["review_required"]:
        add_rule(
            "MODEL-001", "HIGH", "MODEL_RISK_SIGNAL",
            "The calibrated bankruptcy probability is above the locked review threshold.",
        )
    if bool(facts["liabilities_exceed_assets"]):
        add_rule(
            "BAL-001", "HIGH", "BALANCE_SHEET",
            "Reported total liabilities exceed total assets.",
        )
    if bool(facts["negative_equity"]):
        add_rule(
            "BAL-002", "HIGH", "BALANCE_SHEET",
            "Equity relative to total assets is negative.",
        )
    if (
        bool(facts["negative_working_capital"])
        and bool(facts["current_assets_below_short_term_liabilities"])
    ):
        add_rule(
            "LIQ-001", "MEDIUM", "LIQUIDITY",
            "Working capital is negative and current assets are below short-term liabilities.",
        )
    if bool(facts["negative_net_profit"]) and bool(facts["negative_ebit"]):
        add_rule(
            "PROF-001", "MEDIUM", "PROFITABILITY",
            "Both net profit and EBIT relative to total assets are negative.",
        )

    # E. Choose workflow route
    severities = {rule["severity"] for rule in triggered_rules}

    if "BLOCKING" in severities:
        workflow_route = "DOCUMENT_COMPLETION_REQUIRED"
        route_reason   = "The assessment cannot proceed until document-integrity issues are resolved."
    elif "HIGH" in severities or "MEDIUM" in severities:
        workflow_route = "ENHANCED_ANALYST_REVIEW"
        route_reason   = "One or more model, financial, liquidity, or profitability signals require enhanced human review."
    else:
        workflow_route = "STANDARD_ANALYST_REVIEW"
        route_reason   = "No enhanced-review rule was triggered, but normal human analyst review remains required."

    return {
        "borrower_id":    borrower_id,
        "policy_version": POLICY_VERSION,
        "workflow_route": workflow_route,
        "route_reason":   route_reason,
        "human_review_required":      True,
        "automatic_lending_decision": False,
        "required_document_types":  sorted(REQUIRED_DOCUMENT_TYPES),
        "present_document_types":   sorted(present_document_types),
        "missing_document_types":   missing_document_types,
        "duplicate_document_types": duplicate_document_types,
        "model_result":         risk_result,
        "triggered_rule_count": len(triggered_rules),
        "triggered_rules":      triggered_rules,
        "disclaimer": (
            "Prototype workflow routing only. These rules are not NIBC "
            "lending policy and do not constitute an approval or rejection."
        ),
    }


print("apply_credit_policy() defined.")
print(f"  Policy version : {POLICY_VERSION}")
print(f"  Facts loaded   : {len(document_facts_df)} borrowers")
print(f"  Manifest loaded: {len(document_manifest_df)} documents")